<a href="https://colab.research.google.com/github/manishakumari13711/2026-ISYS5002-Labs/blob/main/01-labs/Lab10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas requests beautifulsoup4

In [2]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time

In [4]:
# Identify the scraper
headers = {
    "User-Agent": "Mozilla/5.0 (Educational purpose scraper)",
    "Accept": "text/html,application/xhtml+xml"
}

# Website URL
url = "https://books.toscrape.com/"

# Download the page
response = requests.get(url, headers=headers, timeout=15)

# Check whether the request was successful
if response.status_code == 200:
    print(
        f"Successfully downloaded the page! "
        f"Content length: {len(response.text)} characters"
    )
else:
    print(f"Failed to download the page. Status code: {response.status_code}")

# Display the first 500 characters
print(response.text[:500])

Successfully downloaded the page! Content length: 51294 characters
<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->
    <head>
        <title>
    All products | Books to Scrape - Sandbox
</title>

        <meta http-equiv="content-type" content="text/html; charset=UTF-8" /


In [5]:
# Convert the downloaded HTML into a BeautifulSoup object
soup = BeautifulSoup(response.text, "html.parser")

# Extract the page title
page_title = soup.title.text.strip()

print(f"Page title: {page_title}")

Page title: All products | Books to Scrape - Sandbox


In [6]:
# Find all book containers
book_containers = soup.find_all("article", class_="product_pod")

print(f"Found {len(book_containers)} books on this page")

# Create empty lists
titles = []
prices = []
ratings = []

# Extract information from each book
for book in book_containers:

    # Get book title
    title = book.h3.a["title"]
    titles.append(title)

    # Get book price
    price = book.find("p", class_="price_color").text.strip()
    prices.append(price)

    # Get star rating
    star_rating = book.find("p", class_="star-rating")["class"][1]
    ratings.append(star_rating)

# Display the first five books
for i in range(min(5, len(titles))):
    print(
        f"Book {i + 1}: {titles[i]}, "
        f"Price: {prices[i]}, Rating: {ratings[i]}"
    )

Found 20 books on this page
Book 1: A Light in the Attic, Price: Â£51.77, Rating: Three
Book 2: Tipping the Velvet, Price: Â£53.74, Rating: One
Book 3: Soumission, Price: Â£50.10, Rating: One
Book 4: Sharp Objects, Price: Â£47.82, Rating: Four
Book 5: Sapiens: A Brief History of Humankind, Price: Â£54.23, Rating: Five


In [7]:
# Combine the three lists into a dictionary
books_data = {
    "Title": titles,
    "Price": prices,
    "Rating": ratings
}

# Create a pandas DataFrame
books_df = pd.DataFrame(books_data)

# Display the table
display(books_df)

,Title,Price,Rating
0,A Light in the Attic,Â£51.77,Three
1,Tipping the Velvet,Â£53.74,One
2,Soumission,Â£50.10,One
3,Sharp Objects,Â£47.82,Four
4,Sapiens: A Brief History of Humankind,Â£54.23,Five
5,The Requiem Red,Â£22.65,One
6,The Dirty Little Secrets of Getting Your Dream...,Â£33.34,Four
7,The Coming Woman: A Novel Based on the Life of...,Â£17.93,Three
8,The Boys in the Boat: Nine Americans and Their...,Â£22.60,Four
9,The Black Maria,Â£52.15,One


In [9]:
# Re-initialize the DataFrame to ensure a clean run
books_df = pd.DataFrame(books_data)

# Remove all non-numeric characters (except the decimal point) and convert to float
books_df["Price"] = (
    books_df["Price"]
    .str.replace(r"[^0-9.]", "", regex=True)
    .astype(float)
)

# Convert star ratings into numbers
rating_mapping = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

books_df["Rating"] = books_df["Rating"].map(rating_mapping)

# Display the cleaned data
display(books_df)

,Title,Price,Rating
0,A Light in the Attic,51.77,3
1,Tipping the Velvet,53.74,1
2,Soumission,50.10,1
3,Sharp Objects,47.82,4
4,Sapiens: A Brief History of Humankind,54.23,5
5,The Requiem Red,22.65,1
6,The Dirty Little Secrets of Getting Your Dream...,33.34,4
7,The Coming Woman: A Novel Based on the Life of...,17.93,3
8,The Boys in the Boat: Nine Americans and Their...,22.60,4
9,The Black Maria,52.15,1


In [10]:
# Display basic statistics
print("Basic statistics:")
display(books_df.describe())

# Calculate average price for each rating
avg_price_by_rating = (
    books_df.groupby("Rating")["Price"]
    .mean()
    .sort_index()
)

print("Average price by rating:")
display(avg_price_by_rating)

Basic statistics:


,Price,Rating
count,20.000000,20.000000
mean,38.048500,2.850000
std,15.135231,1.565248
min,13.990000,1.000000
25%,22.637500,1.000000
50%,41.380000,3.000000
75%,51.865000,4.000000
max,57.250000,5.000000


Average price by rating:


,Price
Rating,
1,40.018333
2,36.830000
3,42.316667
4,31.105000
5,39.750000


In [11]:
# Create empty lists to store all books
all_titles = []
all_prices = []
all_ratings = []

# Scrape the first three pages
for page_num in range(1, 4):

    # Construct the correct URL
    if page_num == 1:
        page_url = "https://books.toscrape.com/"
    else:
        page_url = (
            f"https://books.toscrape.com/catalogue/"
            f"page-{page_num}.html"
        )

    # Wait one second between requests
    time.sleep(1)

    try:
        # Download the page
        response = requests.get(
            page_url,
            headers=headers,
            timeout=15
        )

        # Check whether the request was successful
        response.raise_for_status()

        print(f"Successfully scraped page {page_num}")

        # Parse HTML
        soup = BeautifulSoup(response.text, "html.parser")

        # Find all book containers
        book_containers = soup.find_all(
            "article",
            class_="product_pod"
        )

        # Extract information from each book
        for book in book_containers:

            # Extract title
            title = book.h3.a["title"]
            all_titles.append(title)

            # Extract price
            price = book.find(
                "p",
                class_="price_color"
            ).text.strip()
            all_prices.append(price)

            # Extract rating
            rating = book.find(
                "p",
                class_="star-rating"
            )["class"][1]
            all_ratings.append(rating)

        print(f"Books found on page {page_num}: "
              f"{len(book_containers)}")

    except requests.RequestException as error:
        print(f"Failed to scrape page {page_num}: {error}")

# Create a DataFrame containing all successfully scraped books
all_books_df = pd.DataFrame({
    "Title": all_titles,
    "Price": all_prices,
    "Rating": all_ratings
})

print("\nTotal books collected:", len(all_books_df))

# Display the collected books
display(all_books_df)

Successfully scraped page 1
Books found on page 1: 20
Successfully scraped page 2
Books found on page 2: 20
Successfully scraped page 3
Books found on page 3: 20

Total books collected: 60


,Title,Price,Rating
0,A Light in the Attic,Â£51.77,Three
1,Tipping the Velvet,Â£53.74,One
2,Soumission,Â£50.10,One
3,Sharp Objects,Â£47.82,Four
4,Sapiens: A Brief History of Humankind,Â£54.23,Five
5,The Requiem Red,Â£22.65,One
6,The Dirty Little Secrets of Getting Your Dream...,Â£33.34,Four
7,The Coming Woman: A Novel Based on the Life of...,Â£17.93,Three
8,The Boys in the Boat: Nine Americans and Their...,Â£22.60,Four
9,The Black Maria,Â£52.15,One


In [13]:
# Re-initialize the DataFrame to ensure a clean run
all_books_df = pd.DataFrame({
    "Title": all_titles,
    "Price": all_prices,
    "Rating": all_ratings
})

# Remove all non-numeric characters (except the decimal point) and convert to float
all_books_df["Price"] = (
    all_books_df["Price"]
    .str.replace(r"[^0-9.]", "", regex=True)
    .astype(float)
)

# Convert rating words into numbers
rating_mapping = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

all_books_df["Rating"] = all_books_df["Rating"].map(
    rating_mapping
)

# Display basic statistics
print("Basic statistics for all scraped books:")
display(all_books_df.describe())

# Calculate average price by rating
avg_price_by_rating = (
    all_books_df.groupby("Rating")["Price"]
    .mean()
    .sort_index()
)

print("Average price by rating:")
display(avg_price_by_rating)

Basic statistics for all scraped books:


,Price,Rating
count,60.000000,60.000000
mean,35.002667,3.000000
std,14.553082,1.507046
min,12.840000,1.000000
25%,22.040000,1.750000
50%,33.485000,3.000000
75%,50.142500,4.000000
max,57.310000,5.000000


Average price by rating:


,Price
Rating,
1,34.311333
2,33.867500
3,38.808462
4,35.526000
5,32.484286
